# Train Model

In [ ]:
import tensorflow as tf

# ตรวจสอบ GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")


In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
df = pd.read_csv('./csbf3nvm/NVM23_train.txt', delimiter=' ', skiprows=1,
                 names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 10
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build CNN model (no pretrained) -----------
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(1024, activation='relu'),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(7)  # Output: [X, Y, Z, W, P, Q, R]
])

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])

model.summary()

# ----------- 5. Custom callback to log training loss components -----------
class TrainLossLogger(Callback):
    def __init__(self, generator, output_file='loss_CNN_stair_10.csv'):
        super().__init__()
        self.generator = generator
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_x, train_y = next(iter(self.generator))
        y_pred = self.model.predict(train_x)

        x_pred = y_pred[:, :3]
        q_pred = y_pred[:, 3:]
        x_true = train_y[:, :3]
        q_true = train_y[:, 3:]

        q_norm = np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7
        q_true_normed = q_true / q_norm

        position_loss = np.mean(np.square(x_pred - x_true))
        quaternion_loss = np.mean(np.square(q_pred - q_true_normed))
        total_loss = position_loss + 400.0 * quaternion_loss

        self.logs.append({
            'Epoch': epoch + 1,
            'Custom_loss': total_loss,
            'Position_loss': position_loss,
            'Quaternion_loss': quaternion_loss
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: loss={total_loss:.4f}, pos_loss={position_loss:.4f}, quat_loss={quaternion_loss:.4f}")

# ----------- 6. Callbacks -----------
checkpoint = ModelCheckpoint('best_model_cnn_stair_10.h5', monitor='loss', save_best_only=True, verbose=1)
loss_logger = TrainLossLogger(train_generator)

# ----------- 7. Train on all data -----------
history = model.fit(
    train_generator,
    epochs=200,
    callbacks=[checkpoint, loss_logger]
)


In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
df = pd.read_csv('./csbf3nvm/NVM23_train.txt', delimiter=' ', skiprows=1,
                 names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 10
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build CNN model (no pretrained) -----------
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D(2, 2),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Conv2D(512, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(1024, activation='relu'),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(7)  # Output: [X, Y, Z, W, P, Q, R]
])

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])

# ----------- 5. Custom callback to log training loss components -----------
class TrainLossLogger(Callback):
    def __init__(self, generator, output_file='loss_CNN_10.csv'):
        super().__init__()
        self.generator = generator
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_x, train_y = next(iter(self.generator))
        y_pred = self.model.predict(train_x)

        x_pred = y_pred[:, :3]
        q_pred = y_pred[:, 3:]
        x_true = train_y[:, :3]
        q_true = train_y[:, 3:]

        q_norm = np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7
        q_true_normed = q_true / q_norm

        position_loss = np.mean(np.square(x_pred - x_true))
        quaternion_loss = np.mean(np.square(q_pred - q_true_normed))
        total_loss = position_loss + 400.0 * quaternion_loss

        self.logs.append({
            'Epoch': epoch + 1,
            'Custom_loss': total_loss,
            'Position_loss': position_loss,
            'Quaternion_loss': quaternion_loss
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: loss={total_loss:.4f}, pos_loss={position_loss:.4f}, quat_loss={quaternion_loss:.4f}")

# ----------- 6. Callbacks -----------
checkpoint = ModelCheckpoint('best_model_cnn_10.h5', monitor='loss', save_best_only=True, verbose=1)
loss_logger = TrainLossLogger(train_generator)

# ----------- 7. Train on all data -----------
history = model.fit(
    train_generator,
    epochs=200,
    callbacks=[checkpoint, loss_logger]
)


In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
df = pd.read_csv('./csbf3nvm/NVM23_train.txt', delimiter=' ', skiprows=1,
                 names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build model with MobileNetV3 -----------
base_model = MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = True  # Set to True later for fine-tuning

inputs = Input(shape=(224, 224, 3))
x = base_model(inputs, training=True)
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
outputs = Dense(7)(x)  # Output: [X, Y, Z, W, P, Q, R]

model = Model(inputs, outputs)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
# ----------- 5. Custom callback for logging -----------
class TrainLossLogger(Callback):
    def __init__(self, generator, output_file='loss_MobileNetV3_L.csv'):
        super().__init__()
        self.generator = generator
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_x, train_y = next(iter(self.generator))
        y_pred = self.model.predict(train_x)

        x_pred = y_pred[:, :3]
        q_pred = y_pred[:, 3:]
        x_true = train_y[:, :3]
        q_true = train_y[:, 3:]

        q_norm = np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7
        q_true_normed = q_true / q_norm

        position_loss = np.mean(np.square(x_pred - x_true))
        quaternion_loss = np.mean(np.square(q_pred - q_true_normed))
        total_loss = position_loss + 400.0 * quaternion_loss

        self.logs.append({
            'Epoch': epoch + 1,
            'Custom_loss': total_loss,
            'Position_loss': position_loss,
            'Quaternion_loss': quaternion_loss
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: loss={total_loss:.4f}, pos_loss={position_loss:.4f}, quat_loss={quaternion_loss:.4f}")

# ----------- 6. Callbacks -----------
checkpoint = ModelCheckpoint('best_model_mobilenetv3_L.h5', monitor='loss', save_best_only=True, verbose=1)
loss_logger = TrainLossLogger(train_generator)

# ----------- 7. Train the model -----------
history = model.fit(
    train_generator,
    epochs=200,
    callbacks=[checkpoint, loss_logger]
)


In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetV2B0
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
df = pd.read_csv('./csbf3nvm/NVM23_train.txt', delimiter=' ', skiprows=1,
                 names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(224, 224),
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build EfficientNetV2B0 model -----------
base_model = EfficientNetV2B0(include_top=False, input_shape=(224, 224, 3), weights='imagenet')

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(7)(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
# ----------- 5. Custom callback to log training loss components -----------
class TrainLossLogger(Callback):
    def __init__(self, generator, output_file='loss_EfficientNetV2B0.csv'):
        super().__init__()
        self.generator = generator
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_x, train_y = next(iter(self.generator))
        y_pred = self.model.predict(train_x)

        x_pred = y_pred[:, :3]
        q_pred = y_pred[:, 3:]
        x_true = train_y[:, :3]
        q_true = train_y[:, 3:]

        q_norm = np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7
        q_true_normed = q_true / q_norm

        position_loss = np.mean(np.square(x_pred - x_true))
        quaternion_loss = np.mean(np.square(q_pred - q_true_normed))
        total_loss = position_loss + 400.0 * quaternion_loss

        self.logs.append({
            'Epoch': epoch + 1,
            'Custom_loss': total_loss,
            'Position_loss': position_loss,
            'Quaternion_loss': quaternion_loss
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: loss={total_loss:.4f}, pos_loss={position_loss:.4f}, quat_loss={quaternion_loss:.4f}")

# ----------- 6. Callbacks -----------
checkpoint = ModelCheckpoint('best_model_alltrain_efficientnetv2b0.h5', monitor='loss', save_best_only=True, verbose=1)
loss_logger = TrainLossLogger(train_generator)

# ----------- 7. Train on all data (no validation) -----------
history = model.fit(
    train_generator,
    epochs=200,
    callbacks=[checkpoint, loss_logger]
)


In [ ]:
import pandas as pd
import numpy as np
import csv
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, Callback
import tensorflow as tf

# ----------- GPU Configuration -----------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU(s) detected: {[gpu.name for gpu in gpus]}")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU found. Training on CPU.")

# ----------- 1. Load all data for training -----------
df = pd.read_csv('./csbf3nvm/NVM23_train.txt', delimiter=' ', skiprows=1,
                 names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])

datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_dataframe(
    dataframe=df,
    directory='./',
    x_col='ImageFile',
    y_col=['X', 'Y', 'Z', 'W', 'P', 'Q', 'R'],
    target_size=(299, 299),  # InceptionV3 requires 299x299 input
    batch_size=64,
    class_mode='raw',
    shuffle=True
)

# ----------- 2. Define custom loss function -----------
def custom_loss(y_true, y_pred):
    beta = 400.0
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]

    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True)) + K.epsilon()
    q_true_normed = q_true / q_norm

    position_loss = K.mean(K.square(x_pred - x_true), axis=-1)
    quaternion_loss = K.mean(K.square(q_pred - q_true_normed), axis=-1)

    return position_loss + beta * quaternion_loss

# ----------- 3. Build InceptionV3 model -----------
base_model = InceptionV3(include_top=False, input_shape=(299, 299, 3), weights='imagenet')

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(7)(x)

model = Model(inputs=base_model.input, outputs=predictions)

# ----------- 4. Compile model -----------
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=custom_loss,
              metrics=['mae'])
model.summary()
# ----------- 5. Custom callback to log training loss components -----------
class TrainLossLogger(Callback):
    def __init__(self, generator, output_file='loss_InceptionV3.csv'):
        super().__init__()
        self.generator = generator
        self.output_file = output_file
        self.logs = []

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_x, train_y = next(iter(self.generator))
        y_pred = self.model.predict(train_x)

        x_pred = y_pred[:, :3]
        q_pred = y_pred[:, 3:]
        x_true = train_y[:, :3]
        q_true = train_y[:, 3:]

        q_norm = np.linalg.norm(q_true, axis=1, keepdims=True) + 1e-7
        q_true_normed = q_true / q_norm

        position_loss = np.mean(np.square(x_pred - x_true))
        quaternion_loss = np.mean(np.square(q_pred - q_true_normed))
        total_loss = position_loss + 400.0 * quaternion_loss

        self.logs.append({
            'Epoch': epoch + 1,
            'Custom_loss': total_loss,
            'Position_loss': position_loss,
            'Quaternion_loss': quaternion_loss
        })

        pd.DataFrame(self.logs).to_csv(self.output_file, index=False)
        print(f"Epoch {epoch+1}: loss={total_loss:.4f}, pos_loss={position_loss:.4f}, quat_loss={quaternion_loss:.4f}")

# ----------- 6. Callbacks -----------
checkpoint = ModelCheckpoint('best_model_inceptionv3.h5', monitor='loss', save_best_only=True, verbose=1)
loss_logger = TrainLossLogger(train_generator)

# ----------- 7. Train on all data (no validation) -----------
history = model.fit(
    train_generator,
    epochs=200,
    callbacks=[checkpoint, loss_logger]
)


# Validate

In [ ]:
def custom_loss(y_true, y_pred):
    beta = 400
    x_pred = y_pred[:, :3]
    q_pred = y_pred[:, 3:]
    x_true = y_true[:, :3]
    q_true = y_true[:, 3:]
    
    q_norm = K.sqrt(K.sum(K.square(q_true), axis=-1, keepdims=True))
    q_true_normed = q_true / q_norm
    
    return K.mean(K.square(x_pred - x_true), axis=-1) + beta * K.mean(K.square(q_pred - q_true_normed), axis=-1)

In [ ]:
model = tf.keras.models.load_model('./best_model_mobilenetv3_L.h5', custom_objects={'custom_loss': custom_loss})

In [ ]:
model.summary()

In [ ]:
import pandas as pd
import numpy as np

# Assuming test_df and model are already defined as shown in your script
test_df = pd.read_csv('./csbf3nvm/NVM23_filtered_cleaned.txt', delimiter=' ', skiprows=1, names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])
test_df[['X', 'Y', 'Z', 'W', 'P', 'Q', 'R']] = test_df[['X', 'Y', 'Z', 'W', 'P', 'Q', 'R']] 
# Assuming test_datagen has been correctly set up as shown previously
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=None,  # Or your specific directory
    x_col='ImageFile',
    target_size=(224, 224),
    batch_size=1,  # Process images one by one
    class_mode=None,
    shuffle=False
)

# Get predictions
predictions = model.predict(test_generator, steps=len(test_df))

# Combine actual and predicted values
results = test_df.copy()
results[['Pred_X', 'Pred_Y', 'Pred_Z', 'Pred_W', 'Pred_P', 'Pred_Q', 'Pred_R']] = predictions

# Save to CSV
results.to_csv('./result/val_MobileNet_L.csv', index=False)

# Print Mean Squared Error
mse = np.mean((test_df[['X', 'Y', 'Z', 'W', 'P', 'Q', 'R']].values - predictions)**2)
print(f"Mean Squared Error: {mse}")

# Print example outputs for verification
for i, row in results.iterrows():
    print(f"Image: {row['ImageFile']}")
    print(f"Actual   : {row[['X', 'Y', 'Z', 'W', 'P', 'Q', 'R']].values}")
    print(f"Predicted: {row[['Pred_X', 'Pred_Y', 'Pred_Z', 'Pred_W', 'Pred_P', 'Pred_Q', 'Pred_R']].values}\n")


In [ ]:
import pandas as pd
import numpy as np

# Load the test data and predictions
test_df = pd.read_csv('./csbf3nvm/NVM23_filtered_cleaned.txt', delimiter=' ', skiprows=1, names=['ImageFile', 'X', 'Y', 'Z', 'W', 'P', 'Q', 'R'])
predictions = pd.read_csv('./result/val_DeepNav_400_F23S.csv')

# Function to calculate the Euclidean distance
def calculate_positional_error(true_values, pred_values):
    return np.linalg.norm(true_values - pred_values, axis=1)

# Function to calculate the angular error between two quaternions using axis-angle representation
def calculate_angular_error(true_quat, pred_quat):
    # Compute the dot product of the quaternions
    dot_product = np.sum(true_quat * pred_quat, axis=1)
    # Ensure the values are within the valid range for arccos
    dot_product = np.clip(dot_product, -1.0, 1.0)
    # Calculate theta using atan2 for numerical stability
    theta = 2 * np.arctan2(np.linalg.norm(np.cross(true_quat[:, 1:], pred_quat[:, 1:]), axis=1), dot_product)
    # Convert radians to degrees
    return theta * (180.0 / np.pi)

# Calculate positional and angular errors
positional_errors = calculate_positional_error(test_df[['X', 'Y', 'Z']].values, predictions[['Pred_X', 'Pred_Y', 'Pred_Z']].values)
angular_errors = calculate_angular_error(test_df[['W', 'P', 'Q', 'R']].values, predictions[['Pred_W', 'Pred_P', 'Pred_Q', 'Pred_R']].values)

# Calculate Mean Squared Error for positional and angular errors
positional_mse = np.mean(positional_errors)
angular_mse = np.mean(angular_errors)

# Combine errors into a DataFrame
errors_df = pd.DataFrame({
    'Positional Error (m)': positional_errors,
    'Angular Error (degrees)': angular_errors
})

# Print and save the table
print(errors_df)
errors_df.to_csv('./errors_DeepNav_400_S.csv', index=False)

# Print MSE values
print(f"Mean Squared Error (Positional Error): {positional_mse} m^2")
print(f"Mean Squared Error (Angular Error): {angular_mse} degrees^2")
print(positional_mse+ 400*angular_mse)

# Plot Result

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the loss data from CSV files
file_paths = {
    "DeepNav" : "./result/loss_CNN.csv",
    "MobileNetV3" : "./result/loss_MobileNetV3.csv",
    "EfficientNetV2B0": "./result/loss_EfficientNetV2B0.csv",
    "InceptionV3": "./result/loss_InceptionV3.csv"
}

# Load data into dataframes
loss_data = {model: pd.read_csv(file) for model, file in file_paths.items()}

# Define loss types and plotting colors
loss_types = ["Custom_loss", "Position_loss", "Quaternion_loss"]
file_names = ["loss_comparison.png", "position_loss_comparison.png", "quaternion_loss_comparison.png"]
colors = ["blue", "green", "red", "purple"]
models = list(loss_data.keys())

# Create individual plots for each loss type and save at high resolution
for loss_type, file_name in zip(loss_types, file_names):
    plt.figure(figsize=(12, 6))  # wide format for clarity
    for model, color in zip(models, colors):
        plt.plot(loss_data[model]["Epoch"], loss_data[model][loss_type], label=model, color=color, linewidth=2)
    
    plt.title(f"{loss_type} Comparison", fontsize=30)
    plt.xlabel("Epoch", fontsize=30)
    plt.ylabel(loss_type, fontsize=30)
    plt.legend(fontsize=25)
    plt.grid(True)
    plt.tight_layout()
    
    # Save at high resolution
    plt.savefig(file_name, dpi=600, bbox_inches='tight')
    plt.show()


In [ ]:
# Load the actual uploaded CSV files
file_paths = {
    "CNN" : "./new result/loss_23onS_CNN_400.csv",
    "MobileNetV3" : "./new result/loss_values_MobileNetV3.csv",
    "EfficientNetV2B0": "./new result/loss_values_EfficientNetV2B0.csv",
    "InceptionV3": "./new result/loss_values_InceptionV3.csv"
}

# Load the CSV files into DataFrames
loss_data = {model: pd.read_csv(path) for model, path in file_paths.items()}

# Extract final epoch values for each model
final_epoch_values = []
for model, df in loss_data.items():
    last_row = df.iloc[-1]
    final_epoch_values.append({
        "Model": model,
        "Final Loss": last_row["Loss"],
        "Final Position Loss": last_row["Position Loss"],
        "Final Quaternion Loss": last_row["Quaternion Loss"]
    })

df_final_losses = pd.DataFrame(final_epoch_values)

# Display the final epoch loss values
df_final_losses


In [ ]:
# Re-import necessary libraries since execution state was reset
import pandas as pd
import matplotlib.pyplot as plt

# Reload the loss data from CSV files
file_paths = {
    "CNN 400 stair": "final_test/loss_23s_CNN_400.csv",

    "CNN 400 non-stair": "final_test/loss_23nons_CNN_400.csv"
}
# Load data into dataframes
loss_data = {model: pd.read_csv(file) for model, file in file_paths.items()}

# Compute mean values for each loss type in each model
summary_data = {}

for model in loss_data.keys():
    summary_data[model] = {
        "Mean Loss": loss_data[model]["Loss"].mean(),
        "Mean Position Loss": loss_data[model]["Position Loss"].mean(),
        "Mean Quaternion Loss": loss_data[model]["Quaternion Loss"].mean()
    }

# Convert to DataFrame for display
summary_df = pd.DataFrame.from_dict(summary_data, orient="index")

# Display the summary table
print(summary_df)

# Plot loss graphs for each type of loss
fig, axes = plt.subplots(3, 1, figsize=(10, 15))

loss_types = ["Loss", "Position Loss", "Quaternion Loss"]
colors = ["blue", "green"]
models = list(loss_data.keys())

for i, loss_type in enumerate(loss_types):
    ax = axes[i]
    for model, color in zip(models, colors):
        ax.plot(loss_data[model]["Epoch"], loss_data[model][loss_type], label=model, color=color, linewidth=2)
    
    ax.set_title(f"{loss_type} Comparison", fontsize=14)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel(loss_type, fontsize=12)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Re-import necessary libraries since execution state was reset
import pandas as pd


# Define file paths
file_paths = {
    "CNN_10_F23": "./final_test/val_0_CNN_10_F23noS.csv",
    "CNN_120_F23": "./final_test/val_0_CNN_120_F23noS.csv",
    "CNN_400_F23": "./final_test/val_0_CNN_400_F23noS.csv",
}

# Load data into dataframes
loss_data = {model: pd.read_csv(file) for model, file in file_paths.items()}

# Compute mean values for each loss type in each model
summary_data = {}

In [ ]:
import numpy as np

# Compute Mean Squared Error (MSE) for each model
summary_data = {}

for model, df in loss_data.items():
    mse_position = np.mean((df[["X", "Y", "Z"]].values - df[["Pred_X", "Pred_Y", "Pred_Z"]].values) ** 2)
    mse_quaternion = np.mean((df[["W", "P", "Q", "R"]].values - df[["Pred_W", "Pred_P", "Pred_Q", "Pred_R"]].values) ** 2)
    mse_total = mse_position + mse_quaternion  # Total Loss

    summary_data[model] = {
        "Mean Loss": mse_total,
        "Mean Position Loss": mse_position,
        "Mean Quaternion Loss": mse_quaternion
    }

# Convert to DataFrame for display
summary_df = pd.DataFrame.from_dict(summary_data, orient="index")

# Display the summary table
print(summary_df)
